# Version 2 Retrospective 2025 V1-versus-V2 Comparison

This notebook executes GitHub Issue #42 under the committed retrospective protocol. The primary leakage-resistant cohort is the headline result; the secondary operational cohort is a sensitivity view. The 2025 sample is not a new untouched Version 2 holdout, and this notebook cannot declare a final independently temporally validated champion.


## 1. Hard Pre-Execution Gate

Verify the committed protocol, environment, source/model/tokenizer/OOF/policy fingerprints, frozen thresholds, canonical class mapping, parameter finiteness, and Git-ignore boundaries before any 2025 model scoring.


In [1]:
from pathlib import Path
import ast
import gc
import hashlib
import json
import os
import platform
import re
import subprocess
import sys
import time
import warnings

import joblib
import matplotlib.pyplot as plt
import nbformat
import numpy as np
import pandas as pd
import scipy
import sklearn
import torch
import transformers
from scipy.stats import ks_2samp
from sklearn.metrics import accuracy_score, confusion_matrix, precision_recall_fscore_support
from sklearn.model_selection import StratifiedGroupKFold
from transformers import AutoTokenizer, DistilBertForSequenceClassification


def find_project_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / ".git").exists() and (candidate / "reports" / "v2_2025_retrospective_protocol.md").exists():
            return candidate
    raise FileNotFoundError("Project root or committed retrospective protocol not found.")


ROOT = find_project_root(Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.routing_rules import AUTO_ROUTE, HUMAN_REVIEW, route_from_scores

NOTEBOOK_PATH = ROOT / "notebooks" / "11_v2_2025_retrospective_comparison.ipynb"
PROTOCOL_PATH = ROOT / "reports" / "v2_2025_retrospective_protocol.md"
SOURCE_2024 = ROOT / "data" / "processed" / "cfpb_complaints_2024_cleaned.csv"
SOURCE_2025 = ROOT / "data" / "raw" / "cfpb_complaints_2025_raw.csv"
V1_PATH = ROOT / "models" / "best_tfidf_classifier.joblib"
V2_DIR = ROOT / "models" / "v2_distilbert_challenger" / "final"
OOF_PATH = ROOT / "models" / "v2_distilbert_challenger" / "oof" / "development_oof_outputs.npz"
POLICY_PATH = ROOT / "models" / "v1_v2_2024_comparison" / "v2_routing_policy.json"
ISSUE41_OUTPUTS = ROOT / "models" / "v1_v2_2024_comparison" / "final_test_outputs.npz"
ISSUE41_SUMMARY_PATH = ROOT / "models" / "v1_v2_2024_comparison" / "comparison_summary.json"
LOCAL_DIR = ROOT / "models" / "v1_v2_2025_retrospective"
SUMMARY_PATH = LOCAL_DIR / "retrospective_summary.json"
COHORT_PATH = LOCAL_DIR / "cohort_membership.npz"
FIGURE_DIR = ROOT / "reports" / "figures"

PROTOCOL_COMMIT = "48479f389fcd06bfdf4cf3036026f86a8ecb51c7"
PROTOCOL_SHA256 = "22d6989adfe877b862e3609c5a074955b9c7ced9037a2f20102eb361ed8b19fd"
SOURCE_2024_SHA256 = "b115eb0c4a20a881a6a45bfb74cb7d715a726537372baa7d68f09d657cdfd919"
SOURCE_2025_SHA256 = "b59d7842e786f00d6be26b7980a42f67474acb9040db293ddd3641204d25eb3a"
V1_SHA256 = "4514e7e49e305e408e2eaaf296d8607b33e9320547685339eff263e4dda0c94a"
OOF_SHA256 = "72d59db97819d6f06b968520eddac2d0c1d590f36dea4efa627e9c123c1e5b13"
POLICY_SHA256 = "9ca16a8533f26f8e00fd9d57c654af66fa78e21880f7fa7783a9d1adf964d818"
ISSUE41_OUTPUTS_SHA256 = "86e22f40174b3c8f897ad64bf0888064f1f377ef0f7b884b0e8b326c7fae51d2"
CACHE_SHA256 = {
    "primary": "592984def2ee5018140bc3669545c850475afccc78baff8b8c199e5dac87e5ae",
    "secondary": "17d7ab07c16098da344872eaaa65c106611537a55098882fff453dc7da88bea8",
}
TOKEN_CACHE_SHA256 = "1949e2f370a6b45faf78e7dd012f35fe8b3395e14baf6cda7b4f0b526fe463c3"
CACHE_FLOAT_RTOL = 1e-6
CACHE_FLOAT_ATOL = 5e-7
MODEL_REVISION = "12040accade4e8a0f71eabdb258fecc2e7e948be"
MAX_LENGTH = 256
BATCH_SIZE = 16
V1_TOP_THRESHOLD = 0.08
V1_MARGIN_THRESHOLD = 0.73
V2_TOP_THRESHOLD = 0.22
V2_MARGIN_THRESHOLD = 0.91

LABELS = [
    "Checking or savings account",
    "Credit card",
    "Credit reporting or other personal consumer reports",
    "Debt collection",
    "Money transfer, virtual currency, or money service",
    "Mortgage",
    "Student loan",
    "Vehicle loan or lease",
]
LABEL2ID = {label: index for index, label in enumerate(LABELS)}
ID2LABEL = {index: label for label, index in LABEL2ID.items()}

EXPECTED_V2_FILES = {
    "config.json": (1_229, "745d87e88a54bd5bd349b14aae194a1f523189cb6d6f377463419487fa43e370"),
    "model.safetensors": (267_851_024, "e05900579f16e96d75df968cedb71b2b2fde3aae95f1bf73dbe7147306287c23"),
    "special_tokens_map.json": (132, "3c3507f36dff57bce437223db3b3081d1e2b52ec3e56ee55438193ecb2c94dd6"),
    "tokenizer.json": (711_494, "8b79639ec74b46604e730f505186eaafb1006d2fd00f2c4930d168bb7f894680"),
    "tokenizer_config.json": (1_283, "21c3bea73b6711617c657664adcd4d0b02ce20d2db4b2ebdd722a6da8da28bcd"),
    "training_args.bin": (6_033, "568405730a290d2928d67b32870a9a965253e46bc2e9ed516048100a9ba475d8"),
    "training_summary.json": (3_329, "b8a149005ac05614230937c3620fcda2162b9f39c126a1aefa95ef8721a0bfcf"),
    "vocab.txt": (231_508, "07eced375cec144d27c900241f3e339478dec958f92fddbc551f295c992038a3"),
}
APPROVED_TRACKED = {
    "notebooks/11_v2_2025_retrospective_comparison.ipynb",
    "reports/v2_2025_retrospective_results.md",
    "reports/v2_model_card.md",
    "README.md",
    "docs/portfolio_summary.md",
    "reports/figures/v1_v2_2025_primary_confusion_matrices.png",
    "reports/figures/v1_v2_2025_retrospective_comparison.png",
    "reports/figures/v1_v2_2025_routing_comparison.png",
    "reports/figures/v2_2025_signal_token_drift.png",
}

checks = []


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def sha256_array(values: np.ndarray) -> str:
    values = np.ascontiguousarray(values)
    return hashlib.sha256(values.tobytes()).hexdigest()


def git(*arguments: str, check: bool = True) -> str:
    result = subprocess.run(
        ["git", *arguments],
        cwd=ROOT,
        capture_output=True,
        text=True,
        check=check,
    )
    return result.stdout.strip()


def record(name: str, passed: bool, observed, expected) -> None:
    checks.append({
        "check": name,
        "status": "PASS" if bool(passed) else "FAIL",
        "observed": str(observed),
        "expected": str(expected),
    })


def stop_on_failure(section: str) -> None:
    failures = [item for item in checks if item["status"] == "FAIL"]
    if failures:
        display(pd.DataFrame(failures))
        raise RuntimeError(f"{section} failed. Stop before Version 2 2025 scoring.")


execution_commit = git("rev-parse", "HEAD")
protocol_commit = git("log", "-1", "--format=%H", "--", "reports/v2_2025_retrospective_protocol.md")
branch = git("branch", "--show-current")
status_output = subprocess.run(
    ["git", "status", "--short"], cwd=ROOT, capture_output=True, text=True, check=True
).stdout
status_lines = [line for line in status_output.splitlines() if line]
status_paths = {line[3:].replace("\\", "/") for line in status_lines}
unapproved_status = sorted(status_paths.difference(APPROVED_TRACKED))

record("Expected branch or main", branch in {"v2/issue-5-2025-retrospective", "main"}, branch, "issue branch or main")
record("Protocol commit exact", protocol_commit == PROTOCOL_COMMIT, protocol_commit, PROTOCOL_COMMIT)
record("Protocol commit predates or equals execution commit", subprocess.run(["git", "merge-base", "--is-ancestor", protocol_commit, execution_commit], cwd=ROOT).returncode == 0, execution_commit, f"descendant of {protocol_commit}")
record("Protocol SHA-256", sha256_file(PROTOCOL_PATH) == PROTOCOL_SHA256, sha256_file(PROTOCOL_PATH), PROTOCOL_SHA256)
record("Only approved tracked work-tree paths", not unapproved_status, unapproved_status or "none", "none")
record("Python version", platform.python_version() == "3.11.15", platform.python_version(), "3.11.15")
record("Scikit-learn version", sklearn.__version__ == "1.9.0", sklearn.__version__, "1.9.0")
record("PyTorch version", torch.__version__ == "2.9.1+cu126", torch.__version__, "2.9.1+cu126")
record("Transformers version", transformers.__version__ == "4.57.6", transformers.__version__, "4.57.6")
record("CUDA available", torch.cuda.is_available(), torch.cuda.is_available(), True)
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE"
record("Expected GPU", gpu_name == "NVIDIA GeForce GTX 1650", gpu_name, "NVIDIA GeForce GTX 1650")

integrity_specs = {
    "2024 source": (SOURCE_2024, 54_908_639, SOURCE_2024_SHA256),
    "2025 source": (SOURCE_2025, 73_806_040, SOURCE_2025_SHA256),
    "V1 model": (V1_PATH, 3_392_109, V1_SHA256),
    "V2 OOF": (OOF_PATH, 674_788, OOF_SHA256),
    "V2 policy": (POLICY_PATH, 2_368, POLICY_SHA256),
    "Issue 41 final-test outputs": (ISSUE41_OUTPUTS, 407_858, ISSUE41_OUTPUTS_SHA256),
}
for label, (path, expected_size, expected_hash) in integrity_specs.items():
    observed_size = path.stat().st_size if path.is_file() else -1
    observed_hash = sha256_file(path) if path.is_file() else "missing"
    record(f"{label} size", observed_size == expected_size, observed_size, expected_size)
    record(f"{label} SHA-256", observed_hash == expected_hash, observed_hash, expected_hash)

for name, (expected_size, expected_hash) in EXPECTED_V2_FILES.items():
    path = V2_DIR / name
    observed_size = path.stat().st_size if path.is_file() else -1
    observed_hash = sha256_file(path) if path.is_file() else "missing"
    record(f"V2 {name} size", observed_size == expected_size, observed_size, expected_size)
    record(f"V2 {name} SHA-256", observed_hash == expected_hash, observed_hash, expected_hash)

policy = json.loads(POLICY_PATH.read_text(encoding="utf-8"))
selected_policy = policy["selected"]
record("V2 policy top threshold", selected_policy["top_score_threshold"] == V2_TOP_THRESHOLD, selected_policy["top_score_threshold"], V2_TOP_THRESHOLD)
record("V2 policy margin threshold", selected_policy["margin_threshold"] == V2_MARGIN_THRESHOLD, selected_policy["margin_threshold"], V2_MARGIN_THRESHOLD)
record("V2 policy OOF source", policy["oof_sha256"] == OOF_SHA256, policy["oof_sha256"], OOF_SHA256)
record("V2 policy locked before final-test access", policy["final_test_data_loaded_at_lock"] is False, policy["final_test_data_loaded_at_lock"], False)
issue41_summary = json.loads(ISSUE41_SUMMARY_PATH.read_text(encoding="utf-8"))
record("Issue 41 summary fingerprints final-test outputs", issue41_summary["row_output_sha256"] == ISSUE41_OUTPUTS_SHA256, issue41_summary["row_output_sha256"], ISSUE41_OUTPUTS_SHA256)
record("Issue 41 summary records final-test output size", issue41_summary["row_output_size_bytes"] == 407_858, issue41_summary["row_output_size_bytes"], 407_858)

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    v1_model = joblib.load(V1_PATH)
record("V1 loads without warnings", not caught, len(caught), 0)
record("V1 canonical class order", list(v1_model.classes_) == LABELS, list(v1_model.classes_), LABELS)
record("V1 thresholds unchanged", (V1_TOP_THRESHOLD, V1_MARGIN_THRESHOLD) == (0.08, 0.73), (V1_TOP_THRESHOLD, V1_MARGIN_THRESHOLD), (0.08, 0.73))

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    v2_model = DistilBertForSequenceClassification.from_pretrained(V2_DIR, local_files_only=True)
    v2_tokenizer = AutoTokenizer.from_pretrained(V2_DIR, local_files_only=True)
record("V2 reloads without warnings", not caught, len(caught), 0)
record("V2 model class", v2_model.__class__.__name__ == "DistilBertForSequenceClassification", v2_model.__class__.__name__, "DistilBertForSequenceClassification")
record("V2 output labels", v2_model.config.num_labels == 8, v2_model.config.num_labels, 8)
record("V2 label2id", v2_model.config.label2id == LABEL2ID, v2_model.config.label2id, LABEL2ID)
record("V2 id2label", {int(k): v for k, v in v2_model.config.id2label.items()} == ID2LABEL, v2_model.config.id2label, ID2LABEL)
all_parameters_finite = all(torch.isfinite(parameter).all().item() for parameter in v2_model.parameters())
record("V2 parameters finite", all_parameters_finite, all_parameters_finite, True)
v2_model.eval()
with torch.inference_mode():
    synthetic_logits = v2_model(
        input_ids=torch.ones((1, 4), dtype=torch.long),
        attention_mask=torch.ones((1, 4), dtype=torch.long),
    ).logits
record("V2 synthetic logits shape", tuple(synthetic_logits.shape) == (1, 8), tuple(synthetic_logits.shape), (1, 8))
record("V2 synthetic logits finite", torch.isfinite(synthetic_logits).all().item(), torch.isfinite(synthetic_logits).all().item(), True)

LOCAL_DIR.mkdir(parents=True, exist_ok=True)
ignored_checks = []
for relative_path in [
    "models/best_tfidf_classifier.joblib",
    "models/v2_distilbert_challenger/final/model.safetensors",
    "models/v2_distilbert_challenger/oof/development_oof_outputs.npz",
    "models/v1_v2_2024_comparison/v2_routing_policy.json",
    "models/v1_v2_2025_retrospective/probe",
]:
    ignored = subprocess.run(["git", "check-ignore", "-q", relative_path], cwd=ROOT).returncode == 0
    ignored_checks.append(ignored)
record("All local artifact paths Git-ignored", all(ignored_checks), ignored_checks, [True] * len(ignored_checks))

stop_on_failure("Hard pre-execution gate")
gate_table = pd.DataFrame(checks)
display(gate_table)
print(f"Hard pre-execution checks passed: {int(gate_table.status.eq('PASS').sum())}/{len(gate_table)}")
print(f"Protocol commit: {protocol_commit}")
print(f"Protocol SHA-256: {PROTOCOL_SHA256}")
print(f"Execution commit: {execution_commit}")
print(f"Git working-tree entries at notebook start: {len(status_lines)} approved path(s)")
print("Project root located: PASS")
print("No Version 2 2025 scoring has occurred.")


,check,status,observed,expected
0,Expected branch or main,PASS,v2/issue-5-2025-retrospective,issue branch or main
1,Protocol commit exact,PASS,48479f389fcd06bfdf4cf3036026f86a8ecb51c7,48479f389fcd06bfdf4cf3036026f86a8ecb51c7
2,Protocol commit predates or equals execution c...,PASS,48479f389fcd06bfdf4cf3036026f86a8ecb51c7,descendant of 48479f389fcd06bfdf4cf3036026f86a...
3,Protocol SHA-256,PASS,22d6989adfe877b862e3609c5a074955b9c7ced9037a2f...,22d6989adfe877b862e3609c5a074955b9c7ced9037a2f...
4,Only approved tracked work-tree paths,PASS,none,none
5,Python version,PASS,3.11.15,3.11.15
6,Scikit-learn version,PASS,1.9.0,1.9.0
7,PyTorch version,PASS,2.9.1+cu126,2.9.1+cu126
8,Transformers version,PASS,4.57.6,4.57.6
9,CUDA available,PASS,True,True


Hard pre-execution checks passed: 57/57
Protocol commit: 48479f389fcd06bfdf4cf3036026f86a8ecb51c7
Protocol SHA-256: 22d6989adfe877b862e3609c5a074955b9c7ced9037a2f20102eb361ed8b19fd
Execution commit: 48479f389fcd06bfdf4cf3036026f86a8ecb51c7
Git working-tree entries at notebook start: 9 approved path(s)
Project root located: PASS
No Version 2 2025 scoring has occurred.


## 2. Reconstruct the Locked 2025 Cohorts

Reproduce only the committed Version 1 cleaning, overlap, conflict, deduplication, and cohort rules. Display aggregate counts only.


In [2]:
URL_PATTERN = re.compile(r"https?://\S+|www\.\S+", flags=re.IGNORECASE)
WHITESPACE_PATTERN = re.compile(r"\s+")


def normalize_text(value) -> str:
    return " ".join(str(value).strip().split())


def stable_text_hash(value) -> str:
    normalized = normalize_text(value)
    if not normalized:
        raise ValueError("Normalized complaint text must not be empty.")
    return hashlib.sha256(normalized.encode("utf-8")).hexdigest()


def clean_raw_narrative(value) -> str:
    text = "" if pd.isna(value) else str(value)
    return WHITESPACE_PATTERN.sub(" ", URL_PATTERN.sub(" ", text)).strip()


def missing_or_blank(series: pd.Series) -> pd.Series:
    return series.astype("string").str.strip().fillna("").eq("")


source_2024 = pd.read_csv(SOURCE_2024, usecols=["clean_complaint_text", "product"])
record("2024 source rows", len(source_2024) == 50_000, len(source_2024), 50_000)
record("2024 required values complete", int(source_2024.isna().sum().sum()) == 0, int(source_2024.isna().sum().sum()), 0)
source_2024["normalized_text_hash"] = source_2024["clean_complaint_text"].map(stable_text_hash)
conflicting_2024 = set(
    source_2024.groupby("normalized_text_hash", sort=False)["product"].nunique().loc[lambda values: values > 1].index
)
modeling_2024 = source_2024.loc[
    source_2024["product"].isin(LABELS)
    & ~source_2024["normalized_text_hash"].isin(conflicting_2024)
].drop_duplicates(["normalized_text_hash", "product"], keep="first").reset_index(drop=True)

outer_splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
development_indices, final_test_indices = list(
    outer_splitter.split(
        modeling_2024["clean_complaint_text"],
        modeling_2024["product"],
        groups=modeling_2024["normalized_text_hash"],
    )
)[0]
development_groups = set(modeling_2024.iloc[development_indices]["normalized_text_hash"])
final_test_groups = set(modeling_2024.iloc[final_test_indices]["normalized_text_hash"])
record("Corrected 2024 modeling rows", len(modeling_2024) == 33_042, len(modeling_2024), 33_042)
record("2024 development rows", len(development_indices) == 26_433, len(development_indices), 26_433)
record("2024 final-test rows", len(final_test_indices) == 6_609, len(final_test_indices), 6_609)
record("2024 partition overlap", not development_groups.intersection(final_test_groups), len(development_groups.intersection(final_test_groups)), 0)

raw_2025 = pd.read_csv(SOURCE_2025, low_memory=False)
required_2025 = ["complaint_what_happened", "product", "date_received", "complaint_id"]
record("2025 raw rows", len(raw_2025) == 50_000, len(raw_2025), 50_000)
record("2025 raw columns", len(raw_2025.columns) == 17, len(raw_2025.columns), 17)
record("2025 required columns", set(required_2025).issubset(raw_2025.columns), sorted(set(required_2025).difference(raw_2025.columns)) or "present", "present")
blank_masks = {column: missing_or_blank(raw_2025[column]) for column in required_2025}
record("2025 required values complete", sum(int(mask.sum()) for mask in blank_masks.values()) == 0, {key: int(value.sum()) for key, value in blank_masks.items()}, "all zero")

eligible = ~blank_masks["complaint_what_happened"] & ~blank_masks["product"]
cleaned_2025 = raw_2025.loc[eligible, ["complaint_what_happened", "product"]].copy()
cleaned_2025["_source_order"] = np.flatnonzero(eligible.to_numpy())
cleaned_2025["clean_complaint_text"] = cleaned_2025["complaint_what_happened"].map(clean_raw_narrative)
cleaned_2025["product"] = cleaned_2025["product"].astype("string").str.strip()
cleaned_2025 = cleaned_2025.loc[
    cleaned_2025["clean_complaint_text"].str.len().gt(0)
    & cleaned_2025["product"].str.len().gt(0),
    ["_source_order", "clean_complaint_text", "product"],
].reset_index(drop=True)
cleaned_2025["normalized_text_hash"] = cleaned_2025["clean_complaint_text"].map(stable_text_hash)

conflicting_2025 = set(
    cleaned_2025.groupby("normalized_text_hash", sort=False)["product"].nunique().loc[lambda values: values > 1].index
)
secondary_df = cleaned_2025.loc[cleaned_2025["product"].isin(LABELS)].copy().reset_index(drop=True)
after_overlap = secondary_df.loc[
    ~secondary_df["normalized_text_hash"].isin(development_groups.union(final_test_groups))
].copy()
after_conflicts = after_overlap.loc[
    ~after_overlap["normalized_text_hash"].isin(conflicting_2025)
].copy()
primary_df = after_conflicts.drop_duplicates(
    ["normalized_text_hash", "product"], keep="first"
).reset_index(drop=True)

overlap_excluded = len(secondary_df) - len(after_overlap)
conflicts_excluded = len(after_overlap) - len(after_conflicts)
repeats_removed = len(after_conflicts) - len(primary_df)
record("2025 locked-scope rows", len(secondary_df) == 49_225, len(secondary_df), 49_225)
record("2025 out-of-scope rows", len(cleaned_2025) - len(secondary_df) == 775, len(cleaned_2025) - len(secondary_df), 775)
record("Primary cross-year exclusions", overlap_excluded == 5_579, overlap_excluded, 5_579)
record("Primary remaining conflict exclusions", conflicts_excluded == 1_301, conflicts_excluded, 1_301)
record("Primary repeated same-label extras", repeats_removed == 12_189, repeats_removed, 12_189)
record("Primary cohort rows", len(primary_df) == 30_156, len(primary_df), 30_156)
record("Secondary cohort rows", len(secondary_df) == 49_225, len(secondary_df), 49_225)
record("Primary source order unique", primary_df["_source_order"].is_unique, primary_df["_source_order"].is_unique, True)
record("Secondary source order unique", secondary_df["_source_order"].is_unique, secondary_df["_source_order"].is_unique, True)
record("Primary normalized texts unique", primary_df["normalized_text_hash"].is_unique, primary_df["normalized_text_hash"].is_unique, True)
record("Primary no conflicting groups", not primary_df["normalized_text_hash"].isin(conflicting_2025).any(), primary_df["normalized_text_hash"].isin(conflicting_2025).any(), False)
record("Primary no 2024 overlap", not primary_df["normalized_text_hash"].isin(development_groups.union(final_test_groups)).any(), primary_df["normalized_text_hash"].isin(development_groups.union(final_test_groups)).any(), False)
record("All categories in primary", set(primary_df["product"]) == set(LABELS), sorted(set(primary_df["product"])), LABELS)
record("All categories in secondary", set(secondary_df["product"]) == set(LABELS), sorted(set(secondary_df["product"])), LABELS)
record("2025 source still unchanged", SOURCE_2025.stat().st_size == 73_806_040 and sha256_file(SOURCE_2025) == SOURCE_2025_SHA256, (SOURCE_2025.stat().st_size, sha256_file(SOURCE_2025)), (73_806_040, SOURCE_2025_SHA256))
stop_on_failure("Locked cohort reconstruction")

primary_order = primary_df["_source_order"].to_numpy(dtype=np.int64)
secondary_order = secondary_df["_source_order"].to_numpy(dtype=np.int64)
primary_order_sha = sha256_array(primary_order)
secondary_order_sha = sha256_array(secondary_order)
with COHORT_PATH.open("wb") as handle:
    np.savez_compressed(
        handle,
        primary_source_order=primary_order,
        secondary_source_order=secondary_order,
        primary_order_sha256=np.asarray(primary_order_sha),
        secondary_order_sha256=np.asarray(secondary_order_sha),
        protocol_sha256=np.asarray(PROTOCOL_SHA256),
        source_sha256=np.asarray(SOURCE_2025_SHA256),
    )

cohort_flow = pd.DataFrame([
    {"stage": "Raw 2025 source", "excluded": 0, "remaining": 50_000},
    {"stage": "Restrict to locked eight-category scope", "excluded": 775, "remaining": 49_225},
    {"stage": "Exclude cross-year overlap from primary", "excluded": overlap_excluded, "remaining": len(after_overlap)},
    {"stage": "Exclude remaining conflicting-label rows", "excluded": conflicts_excluded, "remaining": len(after_conflicts)},
    {"stage": "Remove repeated same-label extras", "excluded": repeats_removed, "remaining": len(primary_df)},
])
category_counts = pd.DataFrame({
    "category": LABELS,
    "primary_rows": [int((primary_df["product"] == label).sum()) for label in LABELS],
    "secondary_rows": [int((secondary_df["product"] == label).sum()) for label in LABELS],
})
display(cohort_flow)
display(category_counts)
print("Locked cohort reconstruction: PASS")
print("Only aggregate cohort counts are displayed; local membership remains Git-ignored.")


,stage,excluded,remaining
0,Raw 2025 source,0,50000
1,Restrict to locked eight-category scope,775,49225
2,Exclude cross-year overlap from primary,5579,43646
3,Exclude remaining conflicting-label rows,1301,42345
4,Remove repeated same-label extras,12189,30156


,category,primary_rows,secondary_rows
0,Checking or savings account,2165,2216
1,Credit card,2253,2369
2,Credit reporting or other personal consumer re...,17843,35215
3,Debt collection,4484,5664
4,"Money transfer, virtual currency, or money ser...",1507,1848
5,Mortgage,719,720
6,Student loan,541,547
7,Vehicle loan or lease,644,646


Locked cohort reconstruction: PASS
Only aggregate cohort counts are displayed; local membership remains Git-ignored.


## 3. Reproduce Version 1 Before Version 2 Scoring

The frozen Version 1 pipeline and thresholds must reproduce the committed primary and secondary results to four displayed decimal places. Failure stops execution.


In [3]:
def classification_result(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    macro = precision_recall_fscore_support(y_true, y_pred, labels=np.arange(8), average="macro", zero_division=0)
    weighted = precision_recall_fscore_support(y_true, y_pred, labels=np.arange(8), average="weighted", zero_division=0)
    per_category = precision_recall_fscore_support(y_true, y_pred, labels=np.arange(8), average=None, zero_division=0)
    matrix = confusion_matrix(y_true, y_pred, labels=np.arange(8))
    row_sums = matrix.sum(axis=1, keepdims=True)
    normalized = np.divide(matrix, row_sums, out=np.zeros_like(matrix, dtype=float), where=row_sums != 0)
    return {
        "rows": int(len(y_true)),
        "correct": int((y_true == y_pred).sum()),
        "incorrect": int((y_true != y_pred).sum()),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_precision": float(macro[0]),
        "macro_recall": float(macro[1]),
        "macro_f1": float(macro[2]),
        "weighted_precision": float(weighted[0]),
        "weighted_recall": float(weighted[1]),
        "weighted_f1": float(weighted[2]),
        "per_category": [
            {
                "category": LABELS[index],
                "precision": float(per_category[0][index]),
                "recall": float(per_category[1][index]),
                "f1": float(per_category[2][index]),
                "support": int(per_category[3][index]),
            }
            for index in range(8)
        ],
        "confusion_counts": matrix.tolist(),
        "confusion_normalized": normalized.tolist(),
    }


def routing_result(y_true: np.ndarray, y_pred: np.ndarray, auto_mask: np.ndarray, reasons: np.ndarray) -> dict:
    review_mask = ~auto_mask
    correct = y_true == y_pred
    auto_rows = int(auto_mask.sum())
    routed_accuracy = float(correct[auto_mask].mean()) if auto_rows else None
    reason_values, reason_counts = np.unique(reasons[review_mask], return_counts=True)
    reason_map = {str(key): int(value) for key, value in zip(reason_values, reason_counts)}
    per_category = []
    for index, label in enumerate(LABELS):
        category_mask = y_true == index
        category_auto = category_mask & auto_mask
        support = int(category_mask.sum())
        auto_count = int(category_auto.sum())
        accuracy = float(correct[category_auto].mean()) if auto_count else None
        per_category.append({
            "category": label,
            "support": support,
            "auto_routed": auto_count,
            "coverage": auto_count / support if support else None,
            "review_rate": 1.0 - (auto_count / support) if support else None,
            "routed_accuracy": accuracy,
            "misroute_rate": None if accuracy is None else 1.0 - accuracy,
        })
    return {
        "rows": int(len(y_true)),
        "auto_routed": auto_rows,
        "human_review": int(review_mask.sum()),
        "coverage": auto_rows / len(y_true),
        "review_rate": float(review_mask.mean()),
        "correct_routed": int((correct & auto_mask).sum()),
        "incorrect_routed": int((~correct & auto_mask).sum()),
        "routed_accuracy": routed_accuracy,
        "misroute_rate": None if routed_accuracy is None else 1.0 - routed_accuracy,
        "review_reasons": reason_map,
        "per_category": per_category,
    }


secondary_y = secondary_df["product"].map(LABEL2ID).to_numpy(dtype=np.int64)
primary_y = primary_df["product"].map(LABEL2ID).to_numpy(dtype=np.int64)
secondary_position_by_order = pd.Series(np.arange(len(secondary_df)), index=secondary_df["_source_order"])
primary_in_secondary = secondary_position_by_order.loc[primary_df["_source_order"]].to_numpy(dtype=np.int64)

v1_secondary_pred_text = v1_model.predict(secondary_df["clean_complaint_text"])
v1_secondary_pred = np.asarray([LABEL2ID[value] for value in v1_secondary_pred_text], dtype=np.int64)
v1_secondary_scores = np.asarray(v1_model.decision_function(secondary_df["clean_complaint_text"]), dtype=np.float64)
v1_primary_pred = v1_secondary_pred[primary_in_secondary]
v1_primary_scores = v1_secondary_scores[primary_in_secondary]

def v1_route(scores: np.ndarray):
    decisions = [
        route_from_scores(
            LABELS,
            row,
            min_top_score=V1_TOP_THRESHOLD,
            min_score_margin=V1_MARGIN_THRESHOLD,
        )
        for row in scores
    ]
    auto = np.asarray([decision["routing_decision"] == AUTO_ROUTE for decision in decisions], dtype=bool)
    reasons = np.asarray([
        "auto_route" if decision["review_reason"] is None else decision["review_reason"]
        for decision in decisions
    ], dtype="U48")
    return auto, reasons


v1_secondary_auto, v1_secondary_reasons = v1_route(v1_secondary_scores)
v1_primary_auto = v1_secondary_auto[primary_in_secondary]
v1_primary_reasons = v1_secondary_reasons[primary_in_secondary]

v1_primary_classification = classification_result(primary_y, v1_primary_pred)
v1_secondary_classification = classification_result(secondary_y, v1_secondary_pred)
v1_primary_routing = routing_result(primary_y, v1_primary_pred, v1_primary_auto, v1_primary_reasons)
v1_secondary_routing = routing_result(secondary_y, v1_secondary_pred, v1_secondary_auto, v1_secondary_reasons)

expected_v1 = {
    "primary": {
        "rows": 30_156, "accuracy": 0.8315, "macro_f1": 0.7527, "weighted_f1": 0.8306,
        "coverage": 0.7251, "review_rate": 0.2749, "routed_accuracy": 0.9203, "misroute_rate": 0.0797,
    },
    "secondary": {
        "rows": 49_225, "accuracy": 0.8771, "macro_f1": 0.7569, "weighted_f1": 0.8766,
        "coverage": 0.7809, "review_rate": 0.2191, "routed_accuracy": 0.9424, "misroute_rate": 0.0576,
    },
}
for cohort_name, classification, routing in [
    ("primary", v1_primary_classification, v1_primary_routing),
    ("secondary", v1_secondary_classification, v1_secondary_routing),
]:
    expected = expected_v1[cohort_name]
    record(f"V1 {cohort_name} rows reproduce", classification["rows"] == expected["rows"], classification["rows"], expected["rows"])
    for metric in ["accuracy", "macro_f1", "weighted_f1"]:
        record(f"V1 {cohort_name} {metric} reproduces", round(classification[metric], 4) == expected[metric], f"{classification[metric]:.4f}", f"{expected[metric]:.4f}")
    for metric in ["coverage", "review_rate", "routed_accuracy", "misroute_rate"]:
        record(f"V1 {cohort_name} {metric} reproduces", round(routing[metric], 4) == expected[metric], f"{routing[metric]:.4f}", f"{expected[metric]:.4f}")

stop_on_failure("Version 1 reproduction gate")
with (LOCAL_DIR / "v1_2025_outputs.npz").open("wb") as handle:
    np.savez_compressed(
        handle,
        secondary_source_order=secondary_order,
        secondary_true=secondary_y,
        secondary_pred=v1_secondary_pred,
        secondary_scores=v1_secondary_scores.astype(np.float32),
        secondary_auto=v1_secondary_auto,
        primary_source_order=primary_order,
        primary_true=primary_y,
        primary_pred=v1_primary_pred,
        primary_scores=v1_primary_scores.astype(np.float32),
        primary_auto=v1_primary_auto,
    )

v1_gate_table = pd.DataFrame([
    {
        "cohort": name,
        "rows": classification["rows"],
        "accuracy": classification["accuracy"],
        "macro_f1": classification["macro_f1"],
        "weighted_f1": classification["weighted_f1"],
        "coverage": routing["coverage"],
        "review_rate": routing["review_rate"],
        "routed_accuracy": routing["routed_accuracy"],
        "misroute_rate": routing["misroute_rate"],
    }
    for name, classification, routing in [
        ("Primary headline", v1_primary_classification, v1_primary_routing),
        ("Secondary sensitivity", v1_secondary_classification, v1_secondary_routing),
    ]
])
display(v1_gate_table.round(4))
print("VERSION 1 REPRODUCTION GATE: PASS")
print("Version 2 2025 scoring is now authorized under the committed protocol.")


,cohort,rows,accuracy,macro_f1,weighted_f1,coverage,review_rate,routed_accuracy,misroute_rate
0,Primary headline,30156,0.8315,0.7527,0.8306,0.7251,0.2749,0.9203,0.0797
1,Secondary sensitivity,49225,0.8771,0.7569,0.8766,0.7809,0.2191,0.9424,0.0576


VERSION 1 REPRODUCTION GATE: PASS
Version 2 2025 scoring is now authorized under the committed protocol.


## 4. Frozen Version 2 Evaluation

Score the primary cohort first, then the secondary cohort, with the frozen model, tokenizer, and policy. Resumable row-level outputs remain under the Git-ignored models directory.


In [4]:
def cache_metadata(cohort_name: str, source_order: np.ndarray) -> dict:
    return {
        "schema": 1,
        "cohort": cohort_name,
        "rows": int(len(source_order)),
        "source_order_sha256": sha256_array(source_order.astype(np.int64)),
        "protocol_commit": PROTOCOL_COMMIT,
        "protocol_sha256": PROTOCOL_SHA256,
        "source_2025_sha256": SOURCE_2025_SHA256,
        "model_revision": MODEL_REVISION,
        "model_safetensors_sha256": EXPECTED_V2_FILES["model.safetensors"][1],
        "tokenizer_json_sha256": EXPECTED_V2_FILES["tokenizer.json"][1],
        "policy_sha256": POLICY_SHA256,
        "top_threshold": V2_TOP_THRESHOLD,
        "margin_threshold": V2_MARGIN_THRESHOLD,
        "max_length": MAX_LENGTH,
        "batch_size": BATCH_SIZE,
        "padding": "dynamic-longest",
        "precision": "cuda-fp16-autocast",
        "label_order": LABELS,
    }


def atomic_npz(path: Path, **arrays) -> None:
    temporary = path.with_suffix(path.suffix + ".partial-write")
    with temporary.open("wb") as handle:
        np.savez_compressed(handle, **arrays)
    temporary.replace(path)


def load_valid_completed_cache(path: Path, cohort_name: str, metadata: dict, source_order: np.ndarray):
    if not path.is_file():
        raise RuntimeError(f"Completed cache is missing: {path.name}. New inference is disabled.")
    expected_rows = len(source_order)
    required_arrays = {"metadata_json", "source_order", "logits", "completed", "predicted", "top_score", "margin"}
    record(f"V2 {cohort_name} cache SHA-256", sha256_file(path) == CACHE_SHA256[cohort_name], sha256_file(path), CACHE_SHA256[cohort_name])
    with np.load(path, allow_pickle=False) as saved:
        missing_arrays = sorted(required_arrays.difference(saved.files))
        record(f"V2 {cohort_name} cache arrays present", not missing_arrays, missing_arrays or "all present", "all present")
        stop_on_failure(f"V2 {cohort_name} completed cache structure")
        saved_meta = json.loads(str(saved["metadata_json"].item()))
        result = {key: saved[key].copy() for key in saved.files if key != "metadata_json"}

    record(f"V2 {cohort_name} cache metadata", saved_meta == metadata, "exact match" if saved_meta == metadata else "mismatch", "exact match")
    record(f"V2 {cohort_name} cache row order", result["source_order"].shape == (expected_rows,) and np.array_equal(result["source_order"], source_order), sha256_array(result["source_order"]), metadata["source_order_sha256"])
    record(f"V2 {cohort_name} cache completion shape", result["completed"].shape == (expected_rows,), result["completed"].shape, (expected_rows,))
    record(f"V2 {cohort_name} cache complete", bool(result["completed"].all()), int(result["completed"].sum()), expected_rows)
    record(f"V2 {cohort_name} cache logits shape", result["logits"].shape == (expected_rows, 8), result["logits"].shape, (expected_rows, 8))
    record(f"V2 {cohort_name} cache logits finite", np.isfinite(result["logits"]).all(), "finite" if np.isfinite(result["logits"]).all() else "non-finite", "finite")
    record(f"V2 {cohort_name} cached predictions shape", result["predicted"].shape == (expected_rows,), result["predicted"].shape, (expected_rows,))
    record(f"V2 {cohort_name} cached top-score shape", result["top_score"].shape == (expected_rows,), result["top_score"].shape, (expected_rows,))
    record(f"V2 {cohort_name} cached margin shape", result["margin"].shape == (expected_rows,), result["margin"].shape, (expected_rows,))
    record(f"V2 {cohort_name} cached signals finite", np.isfinite(result["top_score"]).all() and np.isfinite(result["margin"]).all(), "finite" if np.isfinite(result["top_score"]).all() and np.isfinite(result["margin"]).all() else "non-finite", "finite")
    stop_on_failure(f"V2 {cohort_name} completed cache validation")

    probabilities = torch.softmax(torch.from_numpy(result["logits"]), dim=1).numpy()
    derived_predicted = probabilities.argmax(axis=1).astype(np.int64)
    sorted_probabilities = np.sort(probabilities, axis=1)
    derived_top_score = sorted_probabilities[:, -1].astype(np.float32)
    derived_margin = (sorted_probabilities[:, -1] - sorted_probabilities[:, -2]).astype(np.float32)
    top_max_abs = float(np.max(np.abs(result["top_score"] - derived_top_score)))
    margin_max_abs = float(np.max(np.abs(result["margin"] - derived_margin)))
    record(f"V2 {cohort_name} cached predictions match logits", np.array_equal(result["predicted"], derived_predicted), "exact match" if np.array_equal(result["predicted"], derived_predicted) else "mismatch", "exact match")
    record(f"V2 {cohort_name} cached top scores match logits", np.allclose(result["top_score"], derived_top_score, rtol=CACHE_FLOAT_RTOL, atol=CACHE_FLOAT_ATOL), f"max abs {top_max_abs:.3g}", f"rtol={CACHE_FLOAT_RTOL:g}, atol={CACHE_FLOAT_ATOL:g}")
    record(f"V2 {cohort_name} cached margins match logits", np.allclose(result["margin"], derived_margin, rtol=CACHE_FLOAT_RTOL, atol=CACHE_FLOAT_ATOL), f"max abs {margin_max_abs:.3g}", f"rtol={CACHE_FLOAT_RTOL:g}, atol={CACHE_FLOAT_ATOL:g}")
    stop_on_failure(f"V2 {cohort_name} cache-derived signal validation")

    result["predicted"] = derived_predicted
    result["top_score"] = derived_top_score
    result["margin"] = derived_margin
    return result


def reuse_completed_cache(cohort_name: str, frame: pd.DataFrame):
    source_order = frame["_source_order"].to_numpy(dtype=np.int64)
    metadata = cache_metadata(cohort_name, source_order)
    final_path = LOCAL_DIR / f"{cohort_name}_v2_outputs.npz"
    cached = load_valid_completed_cache(final_path, cohort_name, metadata, source_order)
    print(f"{cohort_name}: reused completed cache and re-derived predictions/signals from logits ({len(source_order):,} rows)")
    return cached, True


torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
v2_model.to("cuda")
v2_model.eval()

primary_v2, primary_reused = reuse_completed_cache("primary", primary_df)
secondary_v2, secondary_reused = reuse_completed_cache("secondary", secondary_df)

record("V2 primary output complete", bool(primary_v2["completed"].all()), int(primary_v2["completed"].sum()), len(primary_df))
record("V2 secondary output complete", bool(secondary_v2["completed"].all()), int(secondary_v2["completed"].sum()), len(secondary_df))
record("V2 primary row order", np.array_equal(primary_v2["source_order"], primary_order), sha256_array(primary_v2["source_order"]), primary_order_sha)
record("V2 secondary row order", np.array_equal(secondary_v2["source_order"], secondary_order), sha256_array(secondary_v2["source_order"]), secondary_order_sha)
record("V2 logits finite", np.isfinite(primary_v2["logits"]).all() and np.isfinite(secondary_v2["logits"]).all(), "finite" if np.isfinite(primary_v2["logits"]).all() and np.isfinite(secondary_v2["logits"]).all() else "non-finite", "finite")
record("Both completed V2 caches reused", primary_reused and secondary_reused, (primary_reused, secondary_reused), (True, True))
record("V2 model remains finite after cache validation", all(torch.isfinite(parameter).all().item() for parameter in v2_model.parameters()), "checked", "all finite")
stop_on_failure("Frozen Version 2 cache reuse")

peak_allocated_mib = torch.cuda.max_memory_allocated() / 1024**2
peak_reserved_mib = torch.cuda.max_memory_reserved() / 1024**2
print(f"Primary artifact reused: {primary_reused}")
print(f"Secondary artifact reused: {secondary_reused}")
print(f"Peak allocated GPU memory: {peak_allocated_mib:.2f} MiB")
print(f"Peak reserved GPU memory: {peak_reserved_mib:.2f} MiB")
print("Frozen Version 2 cache reuse without new inference: PASS")


primary: reused completed cache and re-derived predictions/signals from logits (30,156 rows)
secondary: reused completed cache and re-derived predictions/signals from logits (49,225 rows)
Primary artifact reused: True
Secondary artifact reused: True
Peak allocated GPU memory: 413.58 MiB
Peak reserved GPU memory: 454.00 MiB
Frozen Version 2 cache reuse without new inference: PASS


## 5. Aggregate Classification, Routing, Signal, and Token Analysis

Calculate the precommitted aggregate metrics and descriptive comparisons. Softmax scores and margins are uncalibrated model signals. Development OOF signals come from fold models; 2024 and 2025 signals come from the frozen final model.


In [5]:
def v2_route(predicted: np.ndarray, top_score: np.ndarray, margin: np.ndarray):
    low_top = top_score < V2_TOP_THRESHOLD
    low_margin = margin < V2_MARGIN_THRESHOLD
    auto = ~(low_top | low_margin)
    reasons = np.full(len(predicted), "auto_route", dtype="U48")
    reasons[low_top & low_margin] = "low_top_score_and_low_score_margin"
    reasons[low_top & ~low_margin] = "low_top_score"
    reasons[~low_top & low_margin] = "low_score_margin"
    return auto, reasons


v2_primary_auto, v2_primary_reasons = v2_route(primary_v2["predicted"], primary_v2["top_score"], primary_v2["margin"])
v2_secondary_auto, v2_secondary_reasons = v2_route(secondary_v2["predicted"], secondary_v2["top_score"], secondary_v2["margin"])
v2_primary_classification = classification_result(primary_y, primary_v2["predicted"])
v2_secondary_classification = classification_result(secondary_y, secondary_v2["predicted"])
v2_primary_routing = routing_result(primary_y, primary_v2["predicted"], v2_primary_auto, v2_primary_reasons)
v2_secondary_routing = routing_result(secondary_y, secondary_v2["predicted"], v2_secondary_auto, v2_secondary_reasons)

results = {
    "v1": {
        "primary": {"classification": v1_primary_classification, "routing": v1_primary_routing},
        "secondary": {"classification": v1_secondary_classification, "routing": v1_secondary_routing},
    },
    "v2": {
        "primary": {"classification": v2_primary_classification, "routing": v2_primary_routing},
        "secondary": {"classification": v2_secondary_classification, "routing": v2_secondary_routing},
    },
}
with np.load(ISSUE41_OUTPUTS, allow_pickle=False) as saved:
    issue41_true = saved["true_labels"].astype(np.int64)
    issue41_v1_predicted = saved["v1_predicted_labels"].astype(np.int64)
    issue41_v1_auto = saved["v1_auto_route"].astype(bool)
    issue41_v2_predicted = saved["v2_predicted_labels"].astype(np.int64)
    issue41_v2_auto = saved["v2_auto_route"].astype(bool)
    v2_2024_top = saved["v2_top_softmax_score"].astype(float)
    v2_2024_margin = saved["v2_top_two_softmax_margin"].astype(float)


def exact_2024_reference(y_true: np.ndarray, y_pred: np.ndarray, auto_mask: np.ndarray) -> dict:
    classification = classification_result(y_true, y_pred)
    reasons = np.where(auto_mask, "auto_route", "human_review")
    routing = routing_result(y_true, y_pred, auto_mask, reasons)
    return {
        **{key: classification[key] for key in ["accuracy", "macro_precision", "macro_recall", "macro_f1", "weighted_precision", "weighted_recall", "weighted_f1"]},
        **{key: routing[key] for key in ["coverage", "review_rate", "routed_accuracy", "misroute_rate"]},
    }


reference_2024 = {
    "v1": exact_2024_reference(issue41_true, issue41_v1_predicted, issue41_v1_auto),
    "v2": exact_2024_reference(issue41_true, issue41_v2_predicted, issue41_v2_auto),
}
expected_2024_display = {
    "v1": {"accuracy": 0.8712, "macro_f1": 0.7671, "weighted_f1": 0.8715, "coverage": 0.7705, "review_rate": 0.2295, "routed_accuracy": 0.9503, "misroute_rate": 0.0497},
    "v2": {"accuracy": 0.8882, "macro_f1": 0.7949, "weighted_f1": 0.8859, "coverage": 0.8177, "review_rate": 0.1823, "routed_accuracy": 0.9476, "misroute_rate": 0.0524},
}
record("Issue 41 reference rows", len(issue41_true) == 6_609, len(issue41_true), 6_609)
for model_name in ["v1", "v2"]:
    machine_reference = {**issue41_summary[f"{model_name}_classification"], **issue41_summary[f"{model_name}_routing"]}
    for metric, displayed in expected_2024_display[model_name].items():
        record(f"{model_name.upper()} 2024 {metric} display reference", round(reference_2024[model_name][metric], 4) == displayed, f"{reference_2024[model_name][metric]:.4f}", f"{displayed:.4f}")
        record(f"{model_name.upper()} 2024 {metric} machine reference", np.isclose(reference_2024[model_name][metric], machine_reference[metric], rtol=0, atol=1e-12), f"{reference_2024[model_name][metric]:.12f}", f"{machine_reference[metric]:.12f}")
stop_on_failure("Full-precision Issue 41 reference validation")

overall_rows = []
for model_name in ["v1", "v2"]:
    for cohort_name in ["primary", "secondary"]:
        classification = results[model_name][cohort_name]["classification"]
        routing = results[model_name][cohort_name]["routing"]
        overall_rows.append({
            "model": model_name.upper(),
            "cohort": cohort_name,
            "rows": classification["rows"],
            **{key: classification[key] for key in ["accuracy", "macro_precision", "macro_recall", "macro_f1", "weighted_precision", "weighted_recall", "weighted_f1"]},
            **{key: routing[key] for key in ["auto_routed", "human_review", "coverage", "review_rate", "correct_routed", "incorrect_routed", "routed_accuracy", "misroute_rate"]},
        })
overall_table = pd.DataFrame(overall_rows)

comparison_rows = []
comparison_metrics = ["accuracy", "macro_f1", "weighted_f1", "coverage", "review_rate", "routed_accuracy", "misroute_rate"]
for model_name in ["v1", "v2"]:
    for cohort_name in ["primary", "secondary"]:
        classification = results[model_name][cohort_name]["classification"]
        routing = results[model_name][cohort_name]["routing"]
        values = {**classification, **routing}
        for metric in comparison_metrics:
            comparison_rows.append({
                "comparison": f"{model_name.upper()} 2025 {cohort_name} minus 2024",
                "metric": metric,
                "difference": values[metric] - reference_2024[model_name][metric],
            })
for cohort_name in ["primary", "secondary"]:
    for metric in comparison_metrics:
        v1_value = ({**results["v1"][cohort_name]["classification"], **results["v1"][cohort_name]["routing"]})[metric]
        v2_value = ({**results["v2"][cohort_name]["classification"], **results["v2"][cohort_name]["routing"]})[metric]
        comparison_rows.append({"comparison": f"2025 {cohort_name} V2 minus V1", "metric": metric, "difference": v2_value - v1_value})
for model_name in ["v1", "v2"]:
    for metric in comparison_metrics:
        primary_value = ({**results[model_name]["primary"]["classification"], **results[model_name]["primary"]["routing"]})[metric]
        secondary_value = ({**results[model_name]["secondary"]["classification"], **results[model_name]["secondary"]["routing"]})[metric]
        comparison_rows.append({"comparison": f"{model_name.upper()} secondary minus primary", "metric": metric, "difference": secondary_value - primary_value})
comparison_table = pd.DataFrame(comparison_rows)

category_classification_rows = []
category_routing_rows = []
for model_name in ["v1", "v2"]:
    for cohort_name in ["primary", "secondary"]:
        for item in results[model_name][cohort_name]["classification"]["per_category"]:
            category_classification_rows.append({"model": model_name.upper(), "cohort": cohort_name, **item})
        for item in results[model_name][cohort_name]["routing"]["per_category"]:
            category_routing_rows.append({"model": model_name.upper(), "cohort": cohort_name, **item})
category_classification_table = pd.DataFrame(category_classification_rows)
category_routing_table = pd.DataFrame(category_routing_rows)

with np.load(OOF_PATH, allow_pickle=False) as saved:
    oof_top = saved["top_softmax_score"].astype(float)
    oof_margin = saved["top_two_softmax_margin"].astype(float)
def signal_distribution(cohort: str, top: np.ndarray, margin: np.ndarray) -> dict:
    low_top = top < V2_TOP_THRESHOLD
    low_margin = margin < V2_MARGIN_THRESHOLD
    def describe(values):
        percentiles = np.percentile(values, [5, 25, 50, 75, 95])
        return {
            "mean": float(np.mean(values)), "std": float(np.std(values, ddof=0)),
            "p05": float(percentiles[0]), "p25": float(percentiles[1]), "median": float(percentiles[2]),
            "p75": float(percentiles[3]), "p95": float(percentiles[4]),
        }
    reason_counts = {
        "low_top_score_and_low_score_margin": int((low_top & low_margin).sum()),
        "low_top_score": int((low_top & ~low_margin).sum()),
        "low_score_margin": int((~low_top & low_margin).sum()),
        "auto_route": int((~low_top & ~low_margin).sum()),
    }
    return {
        "cohort": cohort, "rows": int(len(top)), "top_score": describe(top), "margin": describe(margin),
        "fail_top_share": float(low_top.mean()), "fail_margin_share": float(low_margin.mean()),
        "fail_both_share": float((low_top & low_margin).mean()), "review_reasons": reason_counts,
    }


signal_results = [
    signal_distribution("Development OOF fold-model signals", oof_top, oof_margin),
    signal_distribution("2024 frozen-model shared benchmark", v2_2024_top, v2_2024_margin),
    signal_distribution("2025 primary headline", primary_v2["top_score"], primary_v2["margin"]),
    signal_distribution("2025 secondary sensitivity", secondary_v2["top_score"], secondary_v2["margin"]),
]
signal_ks = {
    "primary_vs_2024_top": float(ks_2samp(primary_v2["top_score"], v2_2024_top).statistic),
    "primary_vs_2024_margin": float(ks_2samp(primary_v2["margin"], v2_2024_margin).statistic),
    "secondary_vs_2024_top": float(ks_2samp(secondary_v2["top_score"], v2_2024_top).statistic),
    "secondary_vs_2024_margin": float(ks_2samp(secondary_v2["margin"], v2_2024_margin).statistic),
}


def token_metadata() -> dict:
    return {
        "schema": 1, "protocol_sha256": PROTOCOL_SHA256, "source_sha256": SOURCE_2025_SHA256,
        "tokenizer_json_sha256": EXPECTED_V2_FILES["tokenizer.json"][1],
        "secondary_order_sha256": secondary_order_sha, "special_tokens": True,
        "padding": False, "truncation": False,
    }


token_path = LOCAL_DIR / "token_lengths.npz"
token_meta = token_metadata()
if token_path.is_file():
    record("Token-length cache SHA-256", sha256_file(token_path) == TOKEN_CACHE_SHA256, sha256_file(token_path), TOKEN_CACHE_SHA256)
    stop_on_failure("Token-length cache fingerprint validation")
    with np.load(token_path, allow_pickle=False) as saved:
        saved_meta = json.loads(str(saved["metadata_json"].item()))
        if saved_meta != token_meta or not np.array_equal(saved["secondary_source_order"], secondary_order):
            raise RuntimeError("Mismatched token-length artifact; refusing reuse.")
        secondary_token_lengths = saved["secondary_token_lengths"].astype(np.int64)
    token_lengths_reused = True
else:
    lengths = []
    for start in range(0, len(secondary_df), 256):
        texts = secondary_df.iloc[start:start + 256]["clean_complaint_text"].tolist()
        encoded = v2_tokenizer(
            texts,
            add_special_tokens=True,
            padding=False,
            truncation=False,
            return_length=True,
            verbose=False,
        )
        lengths.extend(encoded["length"])
        if start == 0 or (start // 256 + 1) % 32 == 0:
            print(f"Token audit: {min(start + 256, len(secondary_df)):,}/{len(secondary_df):,}")
    secondary_token_lengths = np.asarray(lengths, dtype=np.int64)
    atomic_npz(
        token_path,
        metadata_json=np.asarray(json.dumps(token_meta, sort_keys=True)),
        secondary_source_order=secondary_order,
        secondary_token_lengths=secondary_token_lengths,
    )
    token_lengths_reused = False
primary_token_lengths = secondary_token_lengths[primary_in_secondary]


def token_distribution(cohort: str, values: np.ndarray) -> dict:
    percentiles = np.percentile(values, [75, 90, 95, 99])
    return {
        "cohort": cohort, "rows": int(len(values)), "minimum": int(values.min()),
        "mean": float(values.mean()), "std": float(values.std(ddof=0)), "median": float(np.median(values)),
        "p75": float(percentiles[0]), "p90": float(percentiles[1]), "p95": float(percentiles[2]),
        "p99": float(percentiles[3]), "maximum": int(values.max()),
        "at_or_below_256": float((values <= 256).mean()),
        "above_256": float((values > 256).mean()), "above_512": float((values > 512).mean()),
    }


development_token_reference = {
    "cohort": "2024 development", "rows": 26_433, "minimum": 4, "mean": 289.0243,
    "std": 386.8186, "median": 187.0, "p75": 352.0, "p90": 601.0, "p95": 837.4,
    "p99": 1_784.0, "maximum": 8_136, "at_or_below_256": 0.619945,
    "above_256": 0.380055, "above_512": 0.134150,
}
token_results = [
    development_token_reference,
    token_distribution("2025 primary headline", primary_token_lengths),
    token_distribution("2025 secondary sensitivity", secondary_token_lengths),
]

summary = {
    "protocol_commit": PROTOCOL_COMMIT,
    "protocol_sha256": PROTOCOL_SHA256,
    "execution_commit": execution_commit,
    "source_2025_sha256": SOURCE_2025_SHA256,
    "cohorts": {
        "primary_rows": len(primary_df), "secondary_rows": len(secondary_df),
        "primary_order_sha256": primary_order_sha, "secondary_order_sha256": secondary_order_sha,
        "out_of_scope": 775, "cross_year_excluded": overlap_excluded,
        "remaining_conflicts_excluded": conflicts_excluded, "repeated_extras_removed": repeats_removed,
    },
    "thresholds": {"v1_top": V1_TOP_THRESHOLD, "v1_margin": V1_MARGIN_THRESHOLD, "v2_top": V2_TOP_THRESHOLD, "v2_margin": V2_MARGIN_THRESHOLD},
    "results": results,
    "reference_2024": reference_2024,
    "reference_2024_source": {"path": ISSUE41_OUTPUTS.relative_to(ROOT).as_posix(), "sha256": ISSUE41_OUTPUTS_SHA256, "calculation": "recomputed from fingerprinted row-level Issue 41 outputs at full precision"},
    "comparisons": comparison_table.to_dict(orient="records"),
    "category_classification": category_classification_table.to_dict(orient="records"),
    "category_routing": category_routing_table.to_dict(orient="records"),
    "signals": signal_results,
    "signal_ks": signal_ks,
    "tokens": token_results,
    "runtime": {
        "primary_artifact_reused": bool(primary_reused), "secondary_artifact_reused": bool(secondary_reused),
        "token_lengths_reused": bool(token_lengths_reused),
        "peak_allocated_mib": float(peak_allocated_mib), "peak_reserved_mib": float(peak_reserved_mib),
    },
    "restrictions": {
        "models_retrained": False, "thresholds_changed": False, "calibration_performed": False,
        "2026_accessed": False, "row_level_outputs_committed": False,
    },
}
SUMMARY_PATH.write_text(json.dumps(summary, indent=2, sort_keys=True), encoding="utf-8")

display(overall_table.round(4))
display(comparison_table.pivot(index="comparison", columns="metric", values="difference").round(4))
display(category_classification_table.round(4))
display(category_routing_table.round(4))
display(pd.DataFrame(signal_results).drop(columns=["top_score", "margin", "review_reasons"]))
display(pd.DataFrame(token_results).round(4))
print(f"Token-length artifact reused: {token_lengths_reused}")
print("Aggregate classification, routing, signal, and token-length analysis: PASS")


,model,cohort,rows,accuracy,macro_precision,macro_recall,macro_f1,weighted_precision,weighted_recall,weighted_f1,auto_routed,human_review,coverage,review_rate,correct_routed,incorrect_routed,routed_accuracy,misroute_rate
0,V1,primary,30156,0.8315,0.7577,0.7508,0.7527,0.8312,0.8315,0.8306,21867,8289,0.7251,0.2749,20125,1742,0.9203,0.0797
1,V1,secondary,49225,0.8771,0.7573,0.7592,0.7569,0.8770,0.8771,0.8766,38442,10783,0.7809,0.2191,36227,2215,0.9424,0.0576
2,V2,primary,30156,0.8404,0.7899,0.7392,0.7620,0.8364,0.8404,0.8364,22831,7325,0.7571,0.2429,20945,1886,0.9174,0.0826
3,V2,secondary,49225,0.8751,0.7880,0.7351,0.7586,0.8694,0.8751,0.8701,39414,9811,0.8007,0.1993,37171,2243,0.9431,0.0569


metric,accuracy,coverage,macro_f1,misroute_rate,review_rate,routed_accuracy,weighted_f1
comparison,,,,,,,
2025 primary V2 minus V1,0.0089,0.0320,0.0093,0.0029,-0.0320,-0.0029,0.0057
2025 secondary V2 minus V1,-0.0020,0.0197,0.0018,-0.0007,-0.0197,0.0007,-0.0065
V1 2025 primary minus 2024,-0.0397,-0.0453,-0.0145,0.0300,0.0453,-0.0300,-0.0408
V1 2025 secondary minus 2024,0.0059,0.0105,-0.0102,0.0079,-0.0105,-0.0079,0.0052
V1 secondary minus primary,0.0456,0.0558,0.0042,-0.0220,-0.0558,0.0220,0.0460
V2 2025 primary minus 2024,-0.0478,-0.0606,-0.0330,0.0302,0.0606,-0.0302,-0.0495
V2 2025 secondary minus 2024,-0.0130,-0.0170,-0.0363,0.0045,0.0170,-0.0045,-0.0158
V2 secondary minus primary,0.0348,0.0436,-0.0033,-0.0257,-0.0436,0.0257,0.0337


,model,cohort,category,precision,recall,f1,support
0,V1,primary,Checking or savings account,0.7115,0.7917,0.7495,2165
1,V1,primary,Credit card,0.6977,0.7182,0.7078,2253
2,V1,primary,Credit reporting or other personal consumer re...,0.8990,0.9097,0.9043,17843
3,V1,primary,Debt collection,0.7301,0.6889,0.7089,4484
4,V1,primary,"Money transfer, virtual currency, or money ser...",0.8105,0.6530,0.7233,1507
5,V1,primary,Mortgage,0.8248,0.8707,0.8471,719
6,V1,primary,Student loan,0.6821,0.7098,0.6957,541
7,V1,primary,Vehicle loan or lease,0.7063,0.6646,0.6848,644
8,V1,secondary,Checking or savings account,0.7146,0.7942,0.7523,2216
9,V1,secondary,Credit card,0.6750,0.6935,0.6842,2369


,model,cohort,category,support,auto_routed,coverage,review_rate,routed_accuracy,misroute_rate
0,V1,primary,Checking or savings account,2165,1298,0.5995,0.4005,0.9291,0.0709
1,V1,primary,Credit card,2253,1323,0.5872,0.4128,0.8443,0.1557
2,V1,primary,Credit reporting or other personal consumer re...,17843,14717,0.8248,0.1752,0.9643,0.0357
3,V1,primary,Debt collection,4484,2733,0.6095,0.3905,0.7644,0.2356
4,V1,primary,"Money transfer, virtual currency, or money ser...",1507,719,0.4771,0.5229,0.7983,0.2017
5,V1,primary,Mortgage,719,496,0.6898,0.3102,0.9657,0.0343
6,V1,primary,Student loan,541,282,0.5213,0.4787,0.8333,0.1667
7,V1,primary,Vehicle loan or lease,644,299,0.4643,0.5357,0.7826,0.2174
8,V1,secondary,Checking or savings account,2216,1347,0.6079,0.3921,0.9295,0.0705
9,V1,secondary,Credit card,2369,1405,0.5931,0.4069,0.7972,0.2028


,cohort,rows,fail_top_share,fail_margin_share,fail_both_share
0,Development OOF fold-model signals,26433,0.000038,0.244997,0.000038
1,2024 frozen-model shared benchmark,6609,0.000000,0.182327,0.000000
2,2025 primary headline,30156,0.000000,0.242904,0.000000
3,2025 secondary sensitivity,49225,0.000000,0.199309,0.000000


,cohort,rows,minimum,mean,std,median,p75,p90,p95,p99,maximum,at_or_below_256,above_256,above_512
0,2024 development,26433,4,289.0243,386.8186,187.0,352.0,601.0,837.4,1784.00,8136,0.6199,0.3801,0.1342
1,2025 primary headline,30156,9,299.2566,340.1773,207.0,377.0,636.0,855.0,1631.45,7143,0.5885,0.4115,0.1521
2,2025 secondary sensitivity,49225,9,256.2537,306.1399,178.0,313.0,553.0,770.8,1425.00,7143,0.6825,0.3175,0.1156


Token-length artifact reused: True
Aggregate classification, routing, signal, and token-length analysis: PASS


## 6. Aggregate Figures

Create only report-ready aggregate figures. No complaint text, identifier, normalized-text hash, row-level prediction, logit, score, margin, or token length is displayed.


In [6]:
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
figure_paths = {
    "confusion": FIGURE_DIR / "v1_v2_2025_primary_confusion_matrices.png",
    "retrospective": FIGURE_DIR / "v1_v2_2025_retrospective_comparison.png",
    "routing": FIGURE_DIR / "v1_v2_2025_routing_comparison.png",
    "drift": FIGURE_DIR / "v2_2025_signal_token_drift.png",
}
short_labels = [
    "Checking /\nsavings", "Credit\ncard", "Credit\nreporting", "Debt\ncollection",
    "Money\ntransfer", "Mortgage", "Student\nloan", "Vehicle\nloan",
]

fig, axes = plt.subplots(1, 2, figsize=(20, 8))
for axis, title, matrix in [
    (axes[0], "Version 1", np.asarray(v1_primary_classification["confusion_normalized"])),
    (axes[1], "Version 2", np.asarray(v2_primary_classification["confusion_normalized"])),
]:
    image = axis.imshow(matrix, cmap="Blues", vmin=0, vmax=1)
    axis.set_title(f"{title} — 2025 Primary Leakage-Resistant Cohort")
    axis.set_xlabel("Predicted category")
    axis.set_ylabel("Actual category")
    axis.set_xticks(range(8), short_labels, rotation=40, ha="right")
    axis.set_yticks(range(8), short_labels)
    for row in range(8):
        for column in range(8):
            value = matrix[row, column]
            axis.text(column, row, f"{value:.1%}", ha="center", va="center", fontsize=7, color="white" if value > 0.5 else "black")
colorbar_axis = fig.add_axes([0.93, 0.22, 0.015, 0.58])
fig.colorbar(image, cax=colorbar_axis)
fig.suptitle("Retrospective 2025 Primary-Cohort Row-Normalized Confusion Matrices", fontsize=15, fontweight="bold")
fig.subplots_adjust(left=0.10, right=0.91, bottom=0.24, top=0.84, wspace=0.35)
if not figure_paths["confusion"].is_file():
    fig.savefig(figure_paths["confusion"], dpi=220, bbox_inches="tight", facecolor="white")
plt.close(fig)

datasets = ["2024 shared benchmark", "2025 primary headline", "2025 secondary sensitivity"]
classification_metrics = ["accuracy", "macro_f1", "weighted_f1"]
routing_metrics = ["coverage", "review_rate", "routed_accuracy", "misroute_rate"]
fig, axes = plt.subplots(1, 2, figsize=(19, 9))
for axis, metrics, title in [(axes[0], classification_metrics, "Classification"), (axes[1], routing_metrics, "Selective routing")]:
    x = np.arange(len(metrics))
    width = 0.12
    series = []
    for dataset in datasets:
        for model_name in ["v1", "v2"]:
            if dataset == "2024 shared benchmark":
                values = [reference_2024[model_name][metric] for metric in metrics]
            else:
                cohort = "primary" if "primary" in dataset else "secondary"
                combined = {**results[model_name][cohort]["classification"], **results[model_name][cohort]["routing"]}
                values = [combined[metric] for metric in metrics]
            series.append((dataset, model_name, values))
    for index, (dataset, model_name, values) in enumerate(series):
        offset = (index - 2.5) * width
        label = f"{model_name.upper()} — {dataset}"
        bars = axis.bar(x + offset, values, width, label=label)
        axis.bar_label(bars, labels=[f"{value:.1%}" for value in values], fontsize=6, rotation=90, padding=2)
    axis.set_xticks(x, [metric.replace("_", " ").title() for metric in metrics])
    axis.set_ylim(0, 1.04)
    axis.grid(axis="y", alpha=0.25)
    axis.set_title(title, fontweight="bold")
axes[0].set_ylabel("Metric value")
handles, labels_legend = axes[0].get_legend_handles_labels()
fig.legend(handles, labels_legend, loc="lower center", bbox_to_anchor=(0.5, 0.08), ncol=3, frameon=False)
fig.suptitle("Frozen V1 and V2: Shared 2024 Benchmark and Retrospective 2025 Cohorts", fontsize=15, fontweight="bold")
fig.text(0.5, 0.025, "Primary is the headline result; secondary is an operational sensitivity view. Values are descriptive, not causal.", ha="center")
fig.subplots_adjust(bottom=0.25, top=0.88, wspace=0.16)
if not figure_paths["retrospective"].is_file():
    fig.savefig(figure_paths["retrospective"], dpi=220, bbox_inches="tight", facecolor="white")
plt.close(fig)

fig, axes = plt.subplots(2, 2, figsize=(19, 13), sharex=True)
x = np.arange(8)
width = 0.36
for row, cohort in enumerate(["primary", "secondary"]):
    for column, metric in enumerate(["coverage", "misroute_rate"]):
        axis = axes[row, column]
        for model_index, model_name in enumerate(["v1", "v2"]):
            table = category_routing_table.loc[(category_routing_table.model == model_name.upper()) & (category_routing_table.cohort == cohort)]
            values = table[metric].to_numpy(dtype=float)
            bars = axis.bar(x + (model_index - 0.5) * width, values, width, label=model_name.upper())
            axis.bar_label(bars, labels=["NA" if not np.isfinite(value) else f"{value:.1%}" for value in values], fontsize=6, rotation=90, padding=2)
        axis.set_title(f"{cohort.title()} cohort — {metric.replace('_', ' ').title()}")
        axis.set_xticks(x, short_labels, rotation=35, ha="right")
        axis.set_ylim(0, max(1.0, axis.get_ylim()[1]))
        axis.grid(axis="y", alpha=0.25)
        if column == 0:
            axis.set_ylabel("Rate")
axes[0, 0].legend()
fig.suptitle("Retrospective 2025 Category-Level Routing Coverage and Risk", fontsize=15, fontweight="bold")
fig.subplots_adjust(hspace=0.42, top=0.92)
if not figure_paths["routing"].is_file():
    fig.savefig(figure_paths["routing"], dpi=220, bbox_inches="tight", facecolor="white")
plt.close(fig)

signal_labels = [
    "Development\nOOF",
    "2024\nshared benchmark",
    "2025 primary\nheadline",
    "2025 secondary\nsensitivity",
]
fig, axes = plt.subplots(2, 2, figsize=(17, 12))
for axis, field, title in [
    (axes[0, 0], ("top_score", "mean"), "Mean top softmax score"),
    (axes[0, 1], ("margin", "mean"), "Mean top-two softmax margin"),
]:
    values = [item[field[0]][field[1]] for item in signal_results]
    bars = axis.bar(np.arange(len(values)), values)
    axis.set_xticks(np.arange(len(values)), signal_labels)
    axis.bar_label(bars, labels=[f"{value:.4f}" for value in values], padding=3)
    axis.set_title(title)
    axis.grid(axis="y", alpha=0.25)
failure_values = [item["fail_margin_share"] for item in signal_results]
bars = axes[1, 0].bar(np.arange(len(failure_values)), failure_values)
axes[1, 0].set_xticks(np.arange(len(failure_values)), signal_labels)
axes[1, 0].bar_label(bars, labels=[f"{value:.4%}" for value in failure_values], padding=3)
axes[1, 0].set_title("Share failing locked V2 margin threshold")
axes[1, 0].grid(axis="y", alpha=0.25)
token_labels = ["2024\ndevelopment", "2025 primary\nheadline", "2025 secondary\nsensitivity"]
token_truncation = [item["above_256"] for item in token_results]
bars = axes[1, 1].bar(np.arange(3), token_truncation)
axes[1, 1].set_xticks(np.arange(3), token_labels)
axes[1, 1].bar_label(bars, labels=[f"{value:.1%}" for value in token_truncation], padding=3)
axes[1, 1].set_title("Share above locked 256-token maximum")
axes[1, 1].grid(axis="y", alpha=0.25)
fig.suptitle("Version 2 Retrospective Signal and Token-Length Drift", fontsize=15, fontweight="bold")
fig.text(0.5, 0.015, "Scores and margins are uncalibrated signals. Top-score failures were effectively zero: development OOF 1/26,433 (0.0038%); 2024 and both 2025 cohorts 0. Review was driven almost entirely by the locked margin threshold.", ha="center", fontsize=8.5, wrap=True)
fig.subplots_adjust(hspace=0.35, top=0.92, bottom=0.10)
fig.savefig(figure_paths["drift"], dpi=220, bbox_inches="tight", facecolor="white")
plt.close(fig)

figure_manifest = pd.DataFrame([
    {"figure": key, "path": path.relative_to(ROOT).as_posix(), "bytes": path.stat().st_size}
    for key, path in figure_paths.items()
])
display(figure_manifest)
print("Four aggregate retrospective figures created: PASS")


,figure,path,bytes
0,confusion,reports/figures/v1_v2_2025_primary_confusion_m...,312699
1,retrospective,reports/figures/v1_v2_2025_retrospective_compa...,203591
2,routing,reports/figures/v1_v2_2025_routing_comparison.png,270804
3,drift,reports/figures/v2_2025_signal_token_drift.png,233473


Four aggregate retrospective figures created: PASS


## 7. Final Validation

Confirm locked models and policies, equal cohort rows, retrospective language boundaries, aggregate artifacts, Git-ignore protections, and the absence of fitting or 2026 access.


In [7]:
# Static and artifact validation; this cell never trains or scores a model.
notebook_document = nbformat.read(NOTEBOOK_PATH, as_version=4)
forbidden_calls = []
for cell in notebook_document.cells:
    if cell.cell_type != "code":
        continue
    tree = ast.parse(cell.source)
    for node in ast.walk(tree):
        if isinstance(node, ast.Call):
            if isinstance(node.func, ast.Attribute):
                call_name = node.func.attr
            elif isinstance(node.func, ast.Name):
                call_name = node.func.id
            else:
                call_name = ""
            if call_name in {"fit", "fit_transform"}:
                forbidden_calls.append((cell.id, call_name))
record("Notebook contains no fit or fit_transform", not forbidden_calls, forbidden_calls or "none", "none")
record("Protocol file unchanged", subprocess.run(["git", "diff", "--exit-code", "--", "reports/v2_2025_retrospective_protocol.md"], cwd=ROOT, capture_output=True).returncode == 0, "unchanged", "unchanged")
record("Thresholds remain locked", (V1_TOP_THRESHOLD, V1_MARGIN_THRESHOLD, V2_TOP_THRESHOLD, V2_MARGIN_THRESHOLD) == (0.08, 0.73, 0.22, 0.91), (V1_TOP_THRESHOLD, V1_MARGIN_THRESHOLD, V2_TOP_THRESHOLD, V2_MARGIN_THRESHOLD), (0.08, 0.73, 0.22, 0.91))
record("Same primary rows for both models", results["v1"]["primary"]["classification"]["rows"] == results["v2"]["primary"]["classification"]["rows"] == 30_156, (results["v1"]["primary"]["classification"]["rows"], results["v2"]["primary"]["classification"]["rows"]), (30_156, 30_156))
record("Same secondary rows for both models", results["v1"]["secondary"]["classification"]["rows"] == results["v2"]["secondary"]["classification"]["rows"] == 49_225, (results["v1"]["secondary"]["classification"]["rows"], results["v2"]["secondary"]["classification"]["rows"]), (49_225, 49_225))
record("Primary remains headline", summary["cohorts"]["primary_rows"] == 30_156, summary["cohorts"]["primary_rows"], 30_156)
record("Secondary remains sensitivity", summary["cohorts"]["secondary_rows"] == 49_225, summary["cohorts"]["secondary_rows"], 49_225)
record("No retraining recorded", summary["restrictions"]["models_retrained"] is False, summary["restrictions"]["models_retrained"], False)
record("No threshold change recorded", summary["restrictions"]["thresholds_changed"] is False, summary["restrictions"]["thresholds_changed"], False)
record("No calibration recorded", summary["restrictions"]["calibration_performed"] is False, summary["restrictions"]["calibration_performed"], False)
record("No 2026 access recorded", summary["restrictions"]["2026_accessed"] is False, summary["restrictions"]["2026_accessed"], False)
record("All four figures nonempty", all(path.is_file() and path.stat().st_size > 10_000 for path in figure_paths.values()), {key: path.stat().st_size if path.is_file() else 0 for key, path in figure_paths.items()}, "all > 10,000 bytes")
record("Aggregate summary exists", SUMMARY_PATH.is_file() and SUMMARY_PATH.stat().st_size > 5_000, SUMMARY_PATH.stat().st_size if SUMMARY_PATH.is_file() else 0, "> 5,000")
record("Local row-level artifacts ignored", subprocess.run(["git", "check-ignore", "-q", "models/v1_v2_2025_retrospective/retrospective_summary.json"], cwd=ROOT).returncode == 0, "ignored", "ignored")

documentation_paths = [
    ROOT / "reports" / "v2_2025_retrospective_results.md",
    ROOT / "reports" / "v2_model_card.md",
    ROOT / "README.md",
    ROOT / "docs" / "portfolio_summary.md",
]
documentation_ready = all(path.is_file() for path in documentation_paths[:2])
if documentation_ready:
    missing_links = []
    for path in documentation_paths:
        content = path.read_text(encoding="utf-8")
        for target in re.findall(r"!?\[[^\]]*\]\(([^)]+)\)", content):
            if "://" not in target and not target.startswith("#"):
                candidate = (path.parent / target.split("#", 1)[0]).resolve()
                if not candidate.exists():
                    missing_links.append((path.name, target))
    record("Documentation relative links resolve", not missing_links, missing_links or "all resolve", "all resolve")
else:
    print("Documentation validation deferred until approved reports are created.")

stop_on_failure("Final Notebook 11 validation")
final_check_table = pd.DataFrame(checks)
display(final_check_table)
passed = int(final_check_table.status.eq("PASS").sum())
print(f"All Notebook 11 checks passed: {passed}/{len(final_check_table)}")
print("NOTEBOOK 11 RETROSPECTIVE EVALUATION: PASS")
print("Primary cohort is the headline result; secondary cohort is an operational sensitivity view.")
print("No model fitting, calibration, threshold selection, 2026 access, or sensitive row-level display occurred.")


,check,status,observed,expected
0,Expected branch or main,PASS,v2/issue-5-2025-retrospective,issue branch or main
1,Protocol commit exact,PASS,48479f389fcd06bfdf4cf3036026f86a8ecb51c7,48479f389fcd06bfdf4cf3036026f86a8ecb51c7
2,Protocol commit predates or equals execution c...,PASS,48479f389fcd06bfdf4cf3036026f86a8ecb51c7,descendant of 48479f389fcd06bfdf4cf3036026f86a...
3,Protocol SHA-256,PASS,22d6989adfe877b862e3609c5a074955b9c7ced9037a2f...,22d6989adfe877b862e3609c5a074955b9c7ced9037a2f...
4,Only approved tracked work-tree paths,PASS,none,none
...,...,...,...,...
175,No 2026 access recorded,PASS,False,False
176,All four figures nonempty,PASS,"{'confusion': 312699, 'retrospective': 203591,...","all > 10,000 bytes"
177,Aggregate summary exists,PASS,74636,"> 5,000"
178,Local row-level artifacts ignored,PASS,ignored,ignored


All Notebook 11 checks passed: 180/180
NOTEBOOK 11 RETROSPECTIVE EVALUATION: PASS
Primary cohort is the headline result; secondary cohort is an operational sensitivity view.
No model fitting, calibration, threshold selection, 2026 access, or sensitive row-level display occurred.
